In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter, BlockBootstrapSampler

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.demand_mlp import DemandMLPPipeline

MODEL = "mlp"
DATASETS = {
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "out": ROOT / "data" / "M5-walmart" / "panel" / MODEL,
        "hp": ROOT / "data" / "M5-walmart" / "panel" / "mlp-opt" / "best_params.json",
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "out": ROOT / "data" / "predict-future-sales-1c" / "panel" / MODEL,
        "hp": ROOT / "data" / "predict-future-sales-1c" / "panel" / "mlp-opt" / "best_params.json",
        "schema": PanelSchema(category="category"),
    },
}

N_FOLDS = 5
MIN_TRAIN_FRAC = 0.5
N_BOOT = 20
BLOCK_SIZE = 4
SEED = 42


def load_panel(spec):
    panel = pd.read_parquet(spec["path"])
    return panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()

def load_hp(spec):
    path = spec["hp"]
    if not path.exists():
        raise FileNotFoundError(f"run mlp-opt.ipynb first: {path}")
    hp = json.loads(path.read_text())
    hp["hidden"] = tuple(hp["hidden"])
    print("  hp:", hp)
    return hp


def featurize(spec, train_raw, val_raw):
    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    return train, val, feats.control_cols


def fit_mlp(train, val, control_cols, hp):
    mlp = DemandMLPPipeline(
        control_cols,
        hidden=hp["hidden"],
        dropout=hp["dropout"],
        lr=hp["lr"],
        act="gelu",
        weight_decay=1e-5,
        d_store=16,
        huber_delta=1.0,
        seed=SEED,
    )
    metrics, elast = mlp.run(train, val)
    summary = elast.groupby(
        ["store_code", "product_i", "product_j", "kind"], as_index=False
    ).agg(
        elasticity=("elasticity", "mean"),
        mae_val=("y_true_i", lambda s: (s - elast.loc[s.index, "y_hat_i"]).abs().mean()),
        n_val=("week_id", "size"),
    )
    return metrics, elast, summary

def summarize(metrics, summary, **extra):
    own = summary[summary.kind == "own"]
    cross = summary[summary.kind == "cross"]
    row = dict(extra)
    row["model"] = MODEL
    row["n_own"] = len(own)
    row["n_cross"] = len(cross)
    row["own_mean"] = float(own.elasticity.mean()) if len(own) else np.nan
    row["cross_mean"] = float(cross.elasticity.mean()) if len(cross) else np.nan
    row["mae_val"] = float(metrics["mae_val"])
    row["rmse_val"] = float(metrics["rmse_val"])
    return row


def print_fit(metrics, elast, summary):
    own = elast[elast.kind == "own"]
    cross = elast[elast.kind == "cross"]
    print("  mae/rmse/r2", metrics["mae_val"], metrics["rmse_val"], metrics["r2_val"])
    if len(own):
        print("  own  mean/min/max", own.elasticity.mean(), own.elasticity.min(), own.elasticity.max())
    if len(cross):
        print("  cross mean/min/max", cross.elasticity.mean(), cross.elasticity.min(), cross.elasticity.max())
    own_s = summary[summary.kind == "own"]
    cross_s = summary[summary.kind == "cross"]
    if len(own_s):
        print("  own  (store,i) mean/min/max", own_s.elasticity.mean(), own_s.elasticity.min(), own_s.elasticity.max())
    if len(cross_s):
        print("  cross (store,i,j) mean/min/max", cross_s.elasticity.mean(), cross_s.elasticity.min(), cross_s.elasticity.max())


def elasticities_long(summary, dataset, split):
    out = summary.copy()
    out.insert(0, "model", MODEL)
    out.insert(1, "dataset", dataset)
    out.insert(2, "split", split)
    return out


def boot_ci(boots, dataset):
    rows = []
    for col in ("own_mean", "cross_mean"):
        q = boots[col].quantile([0.025, 0.5, 0.975])
        rows.append({
            "model": MODEL, "dataset": dataset, "metric": col,
            "q025": float(q.loc[0.025]),
            "median": float(q.loc[0.5]),
            "q975": float(q.loc[0.975]),
        })
    return pd.DataFrame(rows)


def save_table(df, out_dir, name):
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / name
    df.to_csv(path, index=False)
    print("  wrote", path)


def run_kfold(name, spec):
    panel = load_panel(spec)
    splitter = TemporalSplitter(period_col="week_id")
    folds = splitter.expanding_splits(panel, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC)
    hp = load_hp(spec)
    rows = []
    for k, (train_raw, val_raw) in enumerate(folds, 1):
        print(f"--- {name} fold {k}/{N_FOLDS}  "
              f"weeks {train_raw.week_id.min()}–{train_raw.week_id.max()} | "
              f"{val_raw.week_id.min()}–{val_raw.week_id.max()}")
        train, val, controls = featurize(spec, train_raw, val_raw)
        print("  controls:", len(controls))
        metrics, elast, summary = fit_mlp(train, val, controls, hp)
        print_fit(metrics, elast, summary)
        rows.append(summarize(metrics, summary, dataset=name, fold=k))
    out = pd.DataFrame(rows)
    print(out[["own_mean", "cross_mean", "mae_val", "rmse_val"]].agg(["mean", "std"]))
    save_table(out, spec["out"], "kfold.csv")
    return out


def run_bootstrap(name, spec):
    out_dir = spec["out"]
    panel = load_panel(spec)
    splitter = TemporalSplitter(period_col="week_id")
    train_raw, val_raw = splitter.single_split(panel, train_frac=0.8)
    train, val, controls = featurize(spec, train_raw, val_raw)
    hp = load_hp(spec)

    print(f"--- {name} holdout  train weeks {train_raw.week_id.min()}–{train_raw.week_id.max()} | "
          f"val {val_raw.week_id.min()}–{val_raw.week_id.max()}")
    print("  controls:", len(controls), controls)
    metrics, elast, summary = fit_mlp(train, val, controls)
    print_fit(metrics, elast, summary)
    save_table(elasticities_long(summary, name, "holdout"), out_dir, "holdout_elasticities.csv")

    periods = sorted(train["week_id"].unique())
    sampler = BlockBootstrapSampler(
        period_col="week_id", block_size=BLOCK_SIZE, rng=np.random.default_rng(SEED),
    )
    rows = []
    for b in range(N_BOOT):
        print(f"  boot {b + 1}/{N_BOOT}")
        train_b = sampler.sample(train, periods)
        metrics_b, _, summary_b = fit_mlp(train_b, val, controls, hp)
        rows.append(summarize(metrics_b, summary_b, dataset=name, boot=b))
        save_table(pd.DataFrame(rows), out_dir, "bootstrap.csv")
    boots = pd.DataFrame(rows)
    save_table(boots, out_dir, "bootstrap.csv")
    save_table(boot_ci(boots, name), out_dir, "bootstrap_ci.csv")
    for col in ("own_mean", "cross_mean"):
        q = boots[col].quantile([0.025, 0.5, 0.975])
        print(f"{name} {col}: q025={q.loc[0.025]:.4f}  median={q.loc[0.5]:.4f}  q975={q.loc[0.975]:.4f}")
    return boots

In [ ]:
kfold_tables, boot_tables = {}, {}
for name, spec in DATASETS.items():
    panel = load_panel(spec)
    print(f"\n=== {name} === {panel.shape}  "
          f"products={panel['product_code'].nunique()}  "
          f"stores={panel['store_code'].nunique()}")
    print("\n Start kfold")
    kfold_tables[name] = run_kfold(name, spec)
    print("\n Start bootstrap")
    boot_tables[name] = run_bootstrap(name, spec)